# QAOA for MaxCut

Use the Quantum Approximate Optimization Algorithm to split a graph's vertices into two sets that cut the most edges, optimizing the angles with a pure-NumPy outer loop.

**Objectives:**
- Encode a MaxCut problem on a triangle graph and define the classical cut objective
- Build a $p=1$ QAOA circuit: a cost layer $e^{-i\gamma C}$ alternating with a mixer $e^{-i\beta\sum_q X_q}$
- Estimate the expected cut from measurement counts and grid-search $(\gamma,\beta)$ with NumPy
- Visualize the expected-cut landscape and read off the optimum
- Extend the construction to $p=2$ and watch the circuit deepen

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt
from lib.utils.statevector import statevector

# Use local simulator by default (free, instant)
device = LocalSimulator()

## 1. The MaxCut problem

**MaxCut** asks: color each vertex of a graph black or white so that the number of edges
joining *differently colored* vertices (the "cut") is as large as possible. It is a canonical
NP-hard combinatorial problem, and a natural target for QAOA because its objective is a sum of
two-body terms that maps cleanly onto qubits.

We use the smallest interesting case: a **triangle**, three vertices `0, 1, 2` with edges
`(0,1)`, `(1,2)`, `(2,0)`. You cannot 2-color a triangle perfectly (it has an odd cycle), so at
least one edge stays uncut. The best you can do is split it `1` vs `2` vertices, cutting **2** of
the 3 edges. That value, **2**, is the global optimum we want QAOA to approach.

Each vertex becomes a qubit. A measured bitstring is the coloring: bit `0` = one side, bit `1` =
the other. Remember qcsim is **big-endian** — character `i` of the bitstring is qubit `i`
(qubit `0` is the leftmost, most-significant bit), so we can index the string directly by vertex.

In [ ]:
# Triangle graph: 3 vertices, 3 edges
edges = [(0, 1), (1, 2), (2, 0)]
n_qubits = 3

def cut_value(bitstring, edges):
    """Number of edges whose endpoints get different colors.

    Big-endian convention: bitstring[i] is qubit i is vertex i.
    """
    return sum(1 for u, v in edges if bitstring[u] != bitstring[v])

# Enumerate all 2^3 colorings and their cut values to see the optimum.
for x in range(2 ** n_qubits):
    b = format(x, f"0{n_qubits}b")  # big-endian: b[0] is qubit 0
    print(f"{b}  cut = {cut_value(b, edges)}")

# The maximum cut over all colorings must be exactly 2 for a triangle.
brute_force_max = max(cut_value(format(x, f"0{n_qubits}b"), edges)
                      for x in range(2 ** n_qubits))
print("\nbrute-force MaxCut =", brute_force_max)
assert brute_force_max == 2, "Triangle MaxCut optimum must be 2"
# Proven: no coloring cuts all 3 edges; the best cuts exactly 2.

## 2. One QAOA layer: cost unitary + mixer

QAOA alternates two ingredients, $p$ times. Here is one layer ($p=1$).

**Start** in the uniform superposition by applying `H` to every qubit: every coloring is equally
likely, an unbiased starting point.

**Cost unitary** $e^{-i\gamma C}$ imprints the objective as relative *phases*. MaxCut's cost is a
sum over edges of $Z_u Z_v$ terms, and the standard circuit identity is

$$e^{-i\gamma Z_u Z_v} \;=\; \text{CNOT}(u,v)\,\cdot\, R_z(v, 2\gamma)\,\cdot\,\text{CNOT}(u,v)$$

(up to an irrelevant global phase). The two CNOTs compute the parity of qubits $u,v$ onto qubit
$v$; the $R_z(2\gamma)$ phases that parity; the second CNOT uncomputes it. We apply this for every
edge.

**Mixer** $e^{-i\beta\sum_q X_q}$ is just $R_x(q, 2\beta)$ on every qubit. It spreads amplitude
between colorings so that the phases planted by the cost unitary can constructively grow the good
assignments.

In [ ]:
def qaoa_layer(circ, gamma, beta, edges, n_qubits):
    """Append one QAOA layer (cost unitary then mixer) to circ, in place."""
    # Cost unitary: exp(-i*gamma*Z_u*Z_v) per edge, via CNOT - Rz(2*gamma) - CNOT.
    for u, v in edges:
        circ.cnot(u, v)
        circ.rz(v, 2 * gamma)
        circ.cnot(u, v)
    # Mixer: exp(-i*beta*X) on every qubit.
    for q in range(n_qubits):
        circ.rx(q, 2 * beta)
    return circ

def build_qaoa(gammas, betas, edges, n_qubits):
    """Build a p-layer QAOA circuit. len(gammas) == len(betas) == p."""
    circ = Circuit()
    for q in range(n_qubits):
        circ.h(q)                       # uniform superposition over all colorings
    for gamma, beta in zip(gammas, betas):
        qaoa_layer(circ, gamma, beta, edges, n_qubits)
    return circ

# Build a p=1 circuit at sample angles and inspect it.
demo = build_qaoa([0.8], [0.4], edges, n_qubits)
print(demo)
print("qubits:", demo.qubit_count, " depth:", demo.depth)

### The cost-layer identity is real, not approximate

Before trusting the optimizer, let us *prove* the circuit identity
$\text{CNOT}\cdot R_z(2\gamma)\cdot\text{CNOT} = e^{-i\gamma Z\otimes Z}$ (up to global phase) on
a single edge. We compare the exact state vectors with `statevector(...)` — no sampling, so this is
deterministic. The diagonal operator $e^{-i\gamma Z_u Z_v}$ multiplies each basis state by
$e^{\mp i\gamma}$ depending on the parity of the two bits; the two state vectors must agree up to
one overall complex phase.

In [ ]:
# Compare CNOT-Rz(2g)-CNOT against the analytic exp(-i*gamma*ZZ) on 2 qubits.
gamma = 0.8

# Circuit side: start from a generic superposition so all 4 amplitudes are exercised.
prep = Circuit().h(0).ry(1, 1.1)          # arbitrary nontrivial input state
psi_in = statevector(prep)

circ_zz = Circuit().h(0).ry(1, 1.1).cnot(0, 1).rz(1, 2 * gamma).cnot(0, 1)
psi_circ = statevector(circ_zz)

# Analytic side: apply diag(exp(-i*gamma * z_u*z_v)) with z = (-1)**bit.
# Big-endian: index k -> bits b[0]b[1], qubit 0 is the high bit.
phases = np.array([
    np.exp(-1j * gamma * ((-1) ** ((k >> 1) & 1)) * ((-1) ** (k & 1)))
    for k in range(4)
])
psi_analytic = phases * psi_in

# Equal up to a single global phase: align by the first nonzero amplitude, then compare.
idx = np.argmax(np.abs(psi_circ))
ratio = psi_circ[idx] / psi_analytic[idx]
assert np.isclose(abs(ratio), 1.0), "phase factor must have unit modulus"
assert np.allclose(psi_circ, ratio * psi_analytic, atol=1e-10)
print("Cost-layer identity verified: CNOT-Rz(2g)-CNOT == exp(-i*g*ZZ) up to global phase")
# Proven: the cost layer realizes the intended ZZ phase exactly.

## 3. Estimating the expected cut from measurements

QAOA is a *sampling* algorithm. For a given $(\gamma,\beta)$ we run the circuit for many shots,
read the bitstrings, and average their cut values:

$$\langle C\rangle \;=\; \frac{1}{\text{shots}}\sum_{b} \text{count}(b)\,\cdot\,\text{cut}(b).$$

This is the quantity the classical optimizer maximizes. Because it is a statistical estimate of
the true expectation, it carries shot noise — so later we use a *loose* threshold, never an exact
equality. We keep shots modest (2000) for speed.

In [ ]:
def expected_cut(counts, edges):
    """Monte-Carlo estimate of <C> from a Counter of bitstring -> count."""
    shots = sum(counts.values())
    total = 0.0
    for bitstring, count in counts.items():
        total += count * cut_value(bitstring, edges)
    return total / shots

# Run the demo circuit and estimate its expected cut.
SHOTS = 2000
result = device.run(demo, shots=SHOTS).result()
counts = result.measurement_counts
print("counts:", dict(counts))
print("expected cut at (gamma=0.8, beta=0.4):", round(expected_cut(counts, edges), 4))

# Sanity floor that is true for ANY state: the cut estimate lies in [0, 2] for this triangle.
ec = expected_cut(counts, edges)
assert 0.0 <= ec <= 2.0, "expected cut must lie between 0 and the max cut (2)"
# Proven: the estimator is well-formed and bounded by the problem optimum.

## 4. Grid-searching the angles with NumPy

The classical outer loop. Instead of a black-box optimizer like COBYLA, we sweep a $24\times24$ grid:
$\gamma\in[0,\pi)$ and $\beta\in[0,\pi/2)$ (the natural periods for these single-qubit/edge
rotations on this problem). For each grid point we build the circuit, sample, and record the
expected cut. Then we pick the `argmax` — and follow it with a tiny local refinement, exactly the
spirit of a classical optimizer's final descent.

In [ ]:
gammas = np.linspace(0.0, np.pi, 24, endpoint=False)
betas = np.linspace(0.0, np.pi / 2, 24, endpoint=False)
landscape = np.zeros((len(betas), len(gammas)))  # rows = beta, cols = gamma

for i, beta in enumerate(betas):
    for j, gamma in enumerate(gammas):
        circ = build_qaoa([gamma], [beta], edges, n_qubits)
        res = device.run(circ, shots=SHOTS).result()
        landscape[i, j] = expected_cut(res.measurement_counts, edges)

# argmax over the grid.
bi, bj = np.unravel_index(np.argmax(landscape), landscape.shape)
best_gamma, best_beta = gammas[bj], betas[bi]
best_cut = landscape[bi, bj]
print(f"grid best expected cut = {best_cut:.4f}")
print(f"  at gamma = {best_gamma:.4f}, beta = {best_beta:.4f}")

In [ ]:
# Simple local refinement: re-sample a few points around the grid winner and keep the best.
rng = np.random.default_rng(0)
refined_gamma, refined_beta, refined_cut = best_gamma, best_beta, best_cut
step = (gammas[1] - gammas[0]) / 2
for _ in range(12):
    g = np.clip(refined_gamma + rng.uniform(-step, step), 0, np.pi)
    b = np.clip(refined_beta + rng.uniform(-step, step), 0, np.pi / 2)
    res = device.run(build_qaoa([g], [b], edges, n_qubits), shots=SHOTS).result()
    c = expected_cut(res.measurement_counts, edges)
    if c > refined_cut:
        refined_gamma, refined_beta, refined_cut = g, b, c
print(f"refined expected cut = {refined_cut:.4f} at "
      f"gamma={refined_gamma:.4f}, beta={refined_beta:.4f}")

# Headline claim: p=1 QAOA on the triangle reaches a strong expected cut.
# Loose threshold (1.5) because the estimate carries shot noise; the exact p=1
# optimum is ~1.99, far above this floor, so 2000 shots clears it reliably.
assert best_cut >= 1.5, f"grid best expected cut {best_cut:.3f} should exceed 1.5"
# Proven: optimized p=1 QAOA finds colorings that cut well above 3/2 edges on average.

## 5. Visualizing the expected-cut landscape

The grid we filled in *is* the energy landscape the optimizer navigates. Brighter regions are
better colorings. The peak we marked is where $(\gamma,\beta)$ drive the most amplitude onto the
two optimal cuts (`100`/`011`, `010`/`101`, `001`/`110` are the maximizing splits).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(
    landscape, origin="lower", aspect="auto", cmap="viridis",
    extent=[gammas[0], gammas[-1], betas[0], betas[-1]],
)
ax.scatter([best_gamma], [best_beta], color="red", marker="*", s=220,
           edgecolor="white", label=f"argmax = {best_cut:.2f}")
ax.set_xlabel("gamma")
ax.set_ylabel("beta")
ax.set_title("p=1 QAOA expected cut over (gamma, beta)")
ax.legend(loc="upper right")
fig.colorbar(im, ax=ax, label="expected cut")
plt.tight_layout()
plt.show()

## 6. Going deeper: a $p=2$ circuit

More layers give the optimizer more knobs and, in principle, a higher achievable cut — at the cost
of a deeper circuit and a higher-dimensional angle search. Here we simply *build and run* a $p=2$
circuit to confirm the construction generalizes; we do not exhaustively optimize four angles. Note
how the depth roughly doubles.

In [ ]:
# p=2: two gammas and two betas. Reuse the grid winner for the first layer,
# add a second layer at hand-picked angles, and confirm it builds and runs.
p2 = build_qaoa([best_gamma, 0.5], [best_beta, 0.3], edges, n_qubits)
print("p=2 circuit depth:", p2.depth, " (vs p=1 depth", demo.depth, ")")
assert p2.depth > demo.depth, "p=2 must be deeper than p=1"

res2 = device.run(p2, shots=SHOTS).result()
ec2 = expected_cut(res2.measurement_counts, edges)
print("p=2 expected cut (unoptimized 2nd layer):", round(ec2, 4))
# The 2-layer circuit has 2*p*(edges) + n_qubits-style structure; check qubit count is unchanged.
assert p2.qubit_count == n_qubits, "QAOA does not add qubits per layer"
# Proven: the layered construction scales in depth, not width, and runs end-to-end.

## Exercises

Two exercises to stretch the QAOA construction to new graphs and settings.
Each has tiered hints -- expand only what you need -- and a check cell that
tells you when you have it. The simulator is free, so experiment freely.

### Exercise 1 — MaxCut on a different graph

The triangle's odd cycle forced one edge to stay uncut. Swap it for a
**square** -- four vertices `0, 1, 2, 3` in a ring, edges `(0,1)`, `(1,2)`,
`(2,3)`, `(3,0)`. An even cycle *is* 2-colorable, so its true MaxCut is the
full **4**. Re-run the Section 4 grid search on this graph (four qubits now)
and confirm the optimizer pushes the expected cut well past the triangle's
ceiling of 2.

Define `square_edges`: the four edges of the square. Define `best_cut_square`:
the largest expected cut your grid search finds over `(gamma, beta)`.

<details><summary>Hint 1 — nudge</summary>

Nothing about the QAOA machinery cared that the graph was a triangle -- the
edge list is the only problem-specific input. What has to change so the mixer
and the uniform-superposition layer cover four qubits instead of three?

</details>
<details><summary>Hint 2 — approach</summary>

Reuse `build_qaoa` and `expected_cut` unchanged, passing `square_edges` and a
qubit count of `4`. Copy the Section 4 double loop over `gammas` and `betas`,
sample each `(gamma, beta)`, and track the running maximum in
`best_cut_square`. The same `np.linspace` ranges work.

</details>

In [ ]:
# Exercise 1: run the grid search on a 4-vertex square instead of the triangle.
# Define: square_edges -- the four ring edges, and best_cut_square -- the best
# expected cut found over the (gamma, beta) grid.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert len(square_edges) == 4, "a four-vertex ring has exactly 4 edges"
    _verts = {q for edge in square_edges for q in edge}
    assert _verts == {0, 1, 2, 3}, "the square should use all four vertices 0..3"
    _bf = max(cut_value(format(x, "04b"), square_edges) for x in range(2 ** 4))
    assert _bf == 4, (
        "an even cycle is 2-colorable, so every edge can be cut -- its true MaxCut is 4"
    )
    assert best_cut_square <= 4.0, "the expected cut cannot exceed the true MaxCut of 4"
    assert best_cut_square >= 2.5, (
        "an optimized p=1 grid search on the square should push the expected cut "
        "well above the triangle's ceiling of 2"
    )

### Exercise 2 — Grid resolution vs cost

The Section 4 sweep used a 24x24 grid. A finer grid costs more circuit runs but
locates the optimum more precisely. Run the *same* triangle sweep at two
resolutions -- a coarse 12x12 grid and the full 24x24 grid -- and compare. Does
the coarse grid still clear the 1.5 floor? By how much does the located
`(gamma, beta)` shift between the two?

Define `coarse_best`: the best expected cut from the 12x12 grid. Define
`fine_best`: the best expected cut from the 24x24 grid. Both sweep the original
triangle `edges`.

<details><summary>Hint 1 — nudge</summary>

The p=1 landscape you plotted in Section 5 has a broad peak, not a needle.
What does that suggest about whether a coarser grid can still land near the top
of it?

</details>
<details><summary>Hint 2 — approach</summary>

Wrap the Section 4 grid loop in something you can call twice -- once with
`np.linspace(..., 12, ...)` and once with `24`. For each resolution, keep the
maximum expected cut you observe. Assign the two maxima to `coarse_best` and
`fine_best`, keeping the same `gamma` range `[0, pi)` and `beta` range
`[0, pi/2)`.

</details>

In [ ]:
# Exercise 2: compare a coarse 12x12 grid against the full 24x24 grid on the triangle.
# Define: coarse_best -- best expected cut at resolution 12, and
# fine_best -- best expected cut at resolution 24.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert 0.0 <= coarse_best <= 2.0, "the triangle's cut estimate lives in [0, 2]"
    assert 0.0 <= fine_best <= 2.0, "the triangle's cut estimate lives in [0, 2]"
    assert coarse_best >= 1.5, (
        "even the coarse 12x12 grid should clear the 1.5 floor -- the peak is broad"
    )
    assert fine_best >= 1.5, "the 24x24 grid should comfortably clear the 1.5 floor"

## Summary

- **MaxCut** splits a graph's vertices to cut the most edges; the triangle's optimum is **2**, and
  brute force confirmed no coloring cuts all three edges.
- **One QAOA layer** is a cost unitary $e^{-i\gamma C}$ (built per edge as
  `CNOT - Rz(2*gamma) - CNOT`, which we verified equals $e^{-i\gamma Z\otimes Z}$ exactly) followed
  by a mixer $R_x(q, 2\beta)$ on every qubit, starting from `H` on all qubits.
- The **expected cut** is a Monte-Carlo average over measured bitstrings; it is the objective the
  *classical* loop maximizes, so we treat it with a loose, noise-aware threshold.
- A **NumPy grid search** plus a small local refinement stands in for COBYLA; the
  $24\times24$ landscape peaks near an expected cut of ~2, comfortably above our 1.5 floor.
- **More layers** ($p=2$) deepen the circuit without adding qubits, giving the optimizer more
  angles to find even better colorings.

**Next:** [`06-amplitude-estimation.ipynb`](06-amplitude-estimation.ipynb) — estimate an unknown
probability quadratically faster than classical sampling.